# Submission Report (Phase 1)

This notebook checks that the team submission file is valid for Kaggle format.

It verifies:
- column names and order,
- row count,
- query ID ordering,
- JSON format of `relevant_doc_ids`,
- top-k size consistency.


In [1]:
from pathlib import Path
import json
import shutil

import pandas as pd

BASE_DIR = Path('../..')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs'

TEMPLATE_PATH = DATA_DIR / 'submission.csv'
FINAL_PATH = OUTPUT_DIR / 'solutions_SeaFour.csv'

BM25_PATH = OUTPUT_DIR / 'solutions_SeaFour_bm25.csv'
TFIDF_PATH = OUTPUT_DIR / 'solutions_SeaFour_tfidf.csv'


In [2]:
def validate_submission_file(submission_path: Path, template_path: Path) -> dict:
    template_df = pd.read_csv(template_path)
    submission_df = pd.read_csv(submission_path)

    result = {
        'submission': str(submission_path.name),
        'exists': submission_path.exists(),
        'column_match': list(submission_df.columns) == list(template_df.columns),
        'row_count_match': len(submission_df) == len(template_df),
        'query_id_order_match': (
            submission_df.iloc[:, 0].astype(str).tolist() ==
            template_df.iloc[:, 0].astype(str).tolist()
        ),
        'valid_json_lists': True,
        'min_k': None,
        'max_k': None,
    }

    lengths = []
    try:
        for raw in submission_df.iloc[:, 1].astype(str):
            parsed = json.loads(raw)
            if not isinstance(parsed, list):
                result['valid_json_lists'] = False
                break
            lengths.append(len(parsed))
    except Exception:
        result['valid_json_lists'] = False

    if lengths:
        result['min_k'] = min(lengths)
        result['max_k'] = max(lengths)

    result['is_valid'] = all([
        result['column_match'],
        result['row_count_match'],
        result['query_id_order_match'],
        result['valid_json_lists'],
    ])

    return result


In [3]:
checks = []
for path in [BM25_PATH, TFIDF_PATH, FINAL_PATH]:
    if path.exists():
        checks.append(validate_submission_file(path, TEMPLATE_PATH))
    else:
        checks.append({'submission': path.name, 'exists': False})

pd.DataFrame(checks)


,submission,exists,column_match,row_count_match,query_id_order_match,valid_json_lists,min_k,max_k,is_valid
0,solutions_SeaFour_bm25.csv,True,True,True,True,True,100,100,True
1,solutions_SeaFour_tfidf.csv,True,True,True,True,True,100,100,True
2,solutions_SeaFour.csv,True,True,True,True,True,100,100,True


In [4]:
# If final file is missing, create it from BM25 output.
# This matches the pipeline default FINAL_MODEL='bm25'.
if not FINAL_PATH.exists():
    shutil.copy2(BM25_PATH, FINAL_PATH)

print('Final file exists:', FINAL_PATH.exists())
print('Final path:', FINAL_PATH.resolve())


Final file exists: True
Final path: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/outputs/solutions_SeaFour.csv


In [5]:
final_check = validate_submission_file(FINAL_PATH, TEMPLATE_PATH)
pd.DataFrame([final_check])


,submission,exists,column_match,row_count_match,query_id_order_match,valid_json_lists,min_k,max_k,is_valid
0,solutions_SeaFour.csv,True,True,True,True,True,100,100,True


In [6]:
final_df = pd.read_csv(FINAL_PATH)
final_df.head()


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""6b1a2049-6fc6-429d-a11f-061b10ba3507_96947"",...",?
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""da2e5c00-99e4-4e49-9be8-fd34dbe2aba9_120601""...",?
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",?
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""65a10367-e197-471c-8edc-0a73618172e1_14831"",...",?
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""927af133-bf03-4268-a3bd-94bda5c9da82_118230""...",?


## Result

If `is_valid == True`, then `outputs/solutions_SeaFour.csv` matches Kaggle template requirements
(`submission.csv`) for phase-1 style submission formatting.
